<a href="https://colab.research.google.com/github/anastasiafantasia/SIMA2026/blob/main/l06_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L6 — Unire tabelle

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/main/lezioni/L06_unire/L06_task.ipynb)

## Contesto

Il dataset delle imprese dice in che regione si trova ogni impresa, ma non dice niente delle regioni: macroarea, popolazione. Queste informazioni sono in un altro file, `regioni.xlsx`. Per usarle insieme bisogna **unire** le due tabelle, come fa `CERCA.VERT` in Excel.

**La regola di oggi:** dopo ogni unione, conta le righe. Un'unione sbagliata non dà quasi mai errori.

## 0. Preparazione

Esegui questa cella così com'è.

In [ ]:
import pandas as pd

df = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
regioni = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/regioni.xlsx")
print(df.shape)

(2500, 9)


## Parte 1 — L'unione

### 1. Guarda la seconda tabella

Mostra la tabella `regioni`: quante righe ha? Quale colonna ha in comune con `df`?

🔎 *Da cercare:* `pandas head` · `pandas shape`

In [ ]:
# Mostra le dimensioni (righe, colonne) della tabella regioni
print("Dimensioni di regioni:", regioni.shape)

# Mostra le prime righe per identificare le colonne
display(regioni.head())

# Mostra le colonne di df per un confronto diretto
print("Colonne di df:", df.columns.tolist())

Dimensioni di regioni: (20, 3)


,regione,macroarea,popolazione_migliaia
0,Piemonte,Nord-Ovest,4252
1,Valle d'Aosta,Nord-Ovest,123
2,Lombardia,Nord-Ovest,10020
3,Liguria,Nord-Ovest,1507
4,Trentino-Alto Adige/Suedtirol,Nord-Est,1077


Colonne di df: ['id_impresa', 'ragione_sociale', 'regione', 'settore', 'data_costituzione', 'anno', 'addetti', 'fatturato', 'spesa_rs']


### 2. Unisci

Unisci `df` e `regioni` usando la colonna in comune, e salva il risultato in `unito`. Quante righe ha `unito`? Quante dovrebbe averne?

🔎 *Da cercare:* `pandas merge on`

In [ ]:
# Unisci le due tabelle sulla colonna 'regione'
unito = pd.merge(df, regioni, on='regione')

# Stampa il numero di righe del risultato
print("Numero di righe in 'unito':", len(unito))
print("Numero di righe originario in 'df':", len(df))

Numero di righe in 'unito': 2240
Numero di righe originario in 'df': 2500


In [ ]:
# Mostra le prime righe della tabella 'unito'
display(unito.head())

,id_impresa,ragione_sociale,regione,settore,data_costituzione,anno,addetti,fatturato,spesa_rs,macroarea,popolazione_migliaia
0,IMP0001,Linea Group S.r.l.,Emilia-Romagna,Alloggio e ristorazione,1969-05-19,2019,10.0,1032.9,37.1,Nord-Est,4437
1,IMP0001,Linea Group S.r.l.,Emilia-Romagna,Alloggio e ristorazione,1969-05-19,2020,10.0,749.6,NaN,Nord-Est,4437
2,IMP0001,Linea Group S.r.l.,Emilia-Romagna,Alloggio e ristorazione,1969-05-19,2021,11.0,1094.2,33.9,Nord-Est,4437
3,IMP0001,Linea Group S.r.l.,Emilia-Romagna,Alloggio e ristorazione,1969-05-19,2022,12.0,1299.4,71.9,Nord-Est,4437
4,IMP0001,Linea Group S.r.l.,Emilia-Romagna,Alloggio e ristorazione,1969-05-19,2023,11.0,1304.1,55.8,Nord-Est,4437


*Risposta:* Ne ha unite 2240 invece che 2500 perchè l' unione predefinita richiede corrispondenza esatta.

### 3. Chi è sparito?

Rifai l'unione tenendo **tutte** le righe di `df`, e chiedi a pandas di aggiungere una colonna che dice da quale tabella viene ogni riga. Quali regioni non hanno trovato corrispondenza?

🔎 *Da cercare:* `pandas merge how left` · `pandas merge indicator`

In [ ]:
# Esegui il merge con how='left' e l'indicatore
unito_left = pd.merge(df, regioni, on='regione', how='left', indicator=True)

# Trova le righe che non hanno trovato corrispondenza nella tabella regioni
non_abbinate = unito_left[unito_left['_merge'] == 'left_only']

# Mostra le regioni uniche che non hanno trovato corrispondenza
regioni_mancanti = non_abbinate['regione'].unique()
print("Regioni in 'df' che non hanno trovato corrispondenza:", regioni_mancanti)

Regioni in 'df' che non hanno trovato corrispondenza: ['Emilia-Romagna' 'Trentino-Alto Adige']


*Risposta:* emilia-romagna e trentino-alto adige

### 4. Correggi le chiavi

Correggi i nomi delle due regioni nella tabella `regioni`, in modo che coincidano con quelli di `df`. Poi rifai l'unione (con `how="left"`) e verifica: 2.500 righe, nessuna macroarea mancante.

🔎 *Da cercare:* `pandas replace`

In [ ]:
# Controlliamo come sono scritte le regioni in 'regioni' che assomigliano a quelle mancanti
print("Regioni simili nel dataset regioni:")
print(regioni[regioni['regione'].str.contains('Emilia|Trentino', case=False)]['regione'].tolist())

# Correggiamo i nomi nella tabella 'regioni' per farli combaciare con 'df'
# Aggiunta la virgola corretta per separare gli elementi del dizionario
regioni['regione'] = regioni['regione'].replace({
    'Emilia Romagna': 'Emilia-Romagna',
    'Trentino Alto Adige': 'Trentino-Alto Adige',
    'Trentino-Alto Adige/Suedtirol': 'Trentino-Alto Adige'
})

# Rifacciamo l'unione
unito = pd.merge(df, regioni, on='regione', how='left')

# Verifichiamo il numero di righe e la presenza di valori mancanti in macroarea
print("\nNuovo numero di righe in 'unito':", len(unito))
print("Numero di macroaree mancanti:", unito['macroarea'].isnull().sum())

Regioni simili nel dataset regioni:
['Trentino-Alto Adige/Suedtirol', 'Emilia-Romagna']

Nuovo numero di righe in 'unito': 2500
Numero di macroaree mancanti: 0


In [ ]:
# Troviamo quali regioni hanno ancora valori mancanti (NaN) in macroarea nel dataset unito
regioni_con_nan = unito[unito['macroarea'].isnull()]['regione'].unique()
print("Regioni in 'unito' che hanno ancora macroarea mancante:", regioni_con_nan)

# Vediamo quante righe hanno questo problema
righe_mancanti_conteggio = unito['macroarea'].isnull().sum()
print(f"Numero totale di righe con macroarea mancante: {righe_mancanti_conteggio}")

## Parte 2 — Usare l'unione

### 5. Le macroaree

Calcola il fatturato mediano per macroarea.

🔎 *Da cercare:* `pandas groupby median`

In [ ]:
# Calcola il fatturato mediano per ogni macroarea
fatturato_mediano = unito.groupby('macroarea')['fatturato'].median()

# Mostra il risultato
display(fatturato_mediano)

,fatturato
macroarea,
Centro,1385.75
Isole,1503.10
Nord-Est,1474.95
Nord-Ovest,1372.55
Sud,1301.80


### 6. Quante imprese per abitante?

Vogliamo sapere quante imprese del campione ci sono ogni milione di abitanti, regione per regione.

1. Conta le imprese **diverse** per regione e trasforma il risultato in una tabella: `groupby` mette le regioni nell'**indice** (le etichette delle righe), e `reset_index` le riporta in una colonna.
2. Unisci questa tabella con `regioni`.
3. Calcola imprese ogni milione di abitanti (la popolazione è in migliaia).

🔎 *Da cercare:* `pandas groupby nunique` · `pandas reset_index`

In [ ]:
# 1. Conta le imprese diverse (id_impresa) per regione e usa reset_index
imprese_per_regione = df.groupby('regione')['id_impresa'].nunique().reset_index(name='numero_imprese')

# 2. Unisci la tabella ottenuta con la tabella regioni
analisi_regioni = pd.merge(imprese_per_regione, regioni, on='regione')

# 3. Calcola le imprese ogni milione di abitanti e arrotonda a un decimale
analisi_regioni['imprese_per_milione_di_abitanti'] = ((analisi_regioni['numero_imprese'] * 1000) / analisi_regioni['popolazione_migliaia']).round(1)

# Mostra il risultato ordinato per densità decrescente con gli indici resettati in ordine (0, 1, 2...)
tabella_ordinata = analisi_regioni.sort_values(by='imprese_per_milione_di_abitanti', ascending=False).reset_index(drop=True)
display(tabella_ordinata)

,regione,numero_imprese,macroarea,popolazione_migliaia,imprese_per_milione_di_abitanti
0,Valle d'Aosta,9,Nord-Ovest,123,73.2
1,Umbria,13,Centro,856,15.2
2,Molise,4,Sud,290,13.8
3,Calabria,25,Sud,1846,13.5
4,Trentino-Alto Adige,14,Nord-Est,1077,13.0
5,Abruzzo,15,Sud,1272,11.8
6,Liguria,16,Nord-Ovest,1507,10.6
7,Toscana,36,Centro,3660,9.8
8,Lazio,55,Centro,5715,9.6
9,Puglia,36,Sud,3908,9.2


chiedere se è importante che index resti com'è anche se diventa in ordine sparso o se si può ordinare

In [ ]:
# 1. Prepariamo prima il conteggio delle imprese per regione
imprese_per_regione = df.groupby('regione')['id_impresa'].nunique().reset_index(name='numero_imprese')

# 2. Uniamo questa tabella con regioni (Punto 2 del Punto 6)
analisi_regioni = pd.merge(imprese_per_regione, regioni, on='regione')

# Mostriamo la tabella unita risultante
display(analisi_regioni)

,regione,numero_imprese,macroarea,popolazione_migliaia
0,Abruzzo,15,Sud,1272
1,Basilicata,2,Sud,537
2,Calabria,25,Sud,1846
3,Campania,32,Sud,5594
4,Emilia-Romagna,38,Nord-Est,4437
5,Friuli-Venezia Giulia,8,Nord-Est,1194
6,Lazio,55,Centro,5715
7,Liguria,16,Nord-Ovest,1507
8,Lombardia,79,Nord-Ovest,10020
9,Marche,13,Centro,1487


### 7. Quanti italiani?

Calcola la popolazione totale sommando la colonna `popolazione_migliaia`, prima nella tabella `regioni` e poi nella tabella `unito`. I due numeri sono uguali? Quale è giusto, e perché l'altro è sbagliato?

🔎 *Da cercare:* `pandas sum`

In [ ]:
# Calcola la popolazione totale nella tabella originale 'regioni'
pop_regioni = regioni['popolazione_migliaia'].sum()

# Calcola la popolazione totale nella tabella unita 'unito'
pop_unito = unito['popolazione_migliaia'].sum()

print(f"Popolazione totale in 'regioni' (in migliaia): {pop_regioni:,.0f} ({pop_regioni/1000:.2f} milioni)")
print(f"Popolazione totale in 'unito' (in migliaia): {pop_unito:,.0f} ({pop_unito/1000:.2f} milioni)")

Popolazione totale in 'regioni' (in migliaia): 59,014 (59.01 milioni)
Popolazione totale in 'unito' (in migliaia): 11,638,730 (11638.73 milioni)


```markdown
*Risposta:*
I due numeri **non sono uguali**:
- Popolazione in `regioni`: ~59 milioni di abitanti (corretto).
- Popolazione in `unito`: ~11,6 miliardi di abitanti (sbagliato).

**Quale è giusto e perché?**
La popolazione calcolata sulla tabella **`regioni`** è quella corretta perché ogni regione (e la relativa popolazione) è presente esattamente **una sola volta**.

La popolazione calcolata sulla tabella **`unito`** è sbagliata a causa della duplicazione dovuta al merge: ogni regione (e la sua popolazione) è stata ripetuta per ciascuna delle 2500 imprese del campione residenti in quel territorio. Di conseguenza, sommando su `unito` abbiamo contato gli stessi abitanti moltissime volte.

## Domanda finale

Scrivi **due regole** che seguirai ogni volta che unisci due tabelle.

```markdown
*Risposta:*
1. **Controllare sempre il numero di righe prima e dopo l'unione** per assicurarsi che non siano spariti dati importanti o che non ci siano state duplicazioni inattese.
2. **Utilizzare e verificare il parametro `how` nel codice di merge**: specificare esplicitamente il tipo di unione (come `how='left'`) garantisce di preservare tutte le righe della tabella principale (`df`). Questo permette di identificare facilmente eventuali dati non abbinati (che presenteranno valori `NaN` nelle nuove colonne), a differenza dell'unione predefinita (`how='inner'`) che cancella silenziosamente le righe senza corrispondenza.
```

---
## ⭐ Bonus (per chi ha finito)

### B1. Un controllo automatico

`merge` può controllare da solo che ogni regione compaia **una sola volta** nella tabella `regioni`. Trova il parametro e usalo.

🔎 *Da cercare:* `pandas merge validate many_to_one`

In [ ]:
# Usiamo il parametro validate='many_to_one' (o 'm:1') per assicurarci
# che ogni regione compaia al massimo una volta nella tabella di destra ('regioni')
unito_validato = pd.merge(df, regioni, on='regione', how='left', validate='many_to_one')

# Mostriamo le dimensioni per confermare che l'unione è andata a buon fine
print("Unione validata con successo!")
print("Dimensioni finali:", unito_validato.shape)

Unione validata con successo!
Dimensioni finali: (2500, 11)


### B2. Mettere in fila

Unire non vuol dire solo affiancare colonne: a volte si vogliono mettere **una sotto l'altra** due tabelle con le stesse colonne. Prendi le righe del 2019 e quelle del 2023 come due tabelle separate e rimettile insieme in una sola.

🔎 *Da cercare:* `pandas concat`

In [ ]:
# 1. Creiamo due tabelle separate per il 2019 e il 2023
df_2019 = df[df['anno'] == 2019]
df_2023 = df[df['anno'] == 2023]

# 2. Uniamo le due tabelle verticalmente (una sotto l'altra)
df_unito_anni = pd.concat([df_2019, df_2023], axis=0)

# Verifichiamo il risultato mostrando le righe totali e la distribuzione degli anni
print(f"Righe nel 2019: {len(df_2019)}")
print(f"Righe nel 2023: {len(df_2023)}")
print(f"Righe totali concatenate: {len(df_unito_anni)}")

# Mostriamo una anteprima della tabella completa ottenuta
print("\nAnteprima della tabella completa unita (df_unito_anni):")
display(df_unito_anni.head())

# Mostriamo il conteggio delle righe per anno per conferma
display("\nConteggio delle righe per anno nel nuovo dataframe:")
display(df_unito_anni['anno'].value_counts())

Righe nel 2019: 500
Righe nel 2023: 500
Righe totali concatenate: 1000

Anteprima della tabella completa unita (df_unito_anni):


,id_impresa,ragione_sociale,regione,settore,data_costituzione,anno,addetti,fatturato,spesa_rs
0,IMP0001,Linea Group S.r.l.,Emilia-Romagna,Alloggio e ristorazione,1969-05-19,2019,10.0,1032.9,37.1
5,IMP0002,Euro Commerciale S.r.l.s.,Lazio,Alloggio e ristorazione,2004-02-15,2019,7.0,764.6,72.9
10,IMP0003,Nova Servizi S.p.A.,Piemonte,Alloggio e ristorazione,1989-06-28,2019,6.0,1328.9,9.8
15,IMP0004,Euro S.r.l.s.,Marche,Commercio,1983-06-27,2019,6.0,844.8,11.1
20,IMP0005,Nova Group S.r.l.,Lombardia,Alloggio e ristorazione,1993-06-14,2019,25.0,6728.8,46.0


'\nConteggio delle righe per anno nel nuovo dataframe:'

,count
anno,
2019,500
2023,500


### B3. Disegnalo

Disegna un grafico a barre orizzontali delle imprese ogni milione di abitanti, per regione, ordinato.

🔎 *Da cercare:* `pandas plot barh`

In [ ]:
import plotly.express as px

# Ordiniamo i dati in modo crescente per visualizzare la barra più lunga in alto
dati_grafico = tabella_ordinata.sort_values(by='imprese_per_milione_di_abitanti', ascending=True)

# Creiamo il grafico interattivo con Plotly Express
fig = px.bar(
    dati_grafico,
    x='imprese_per_milione_di_abitanti',
    y='regione',
    orientation='h',
    title='Numero di imprese per milione di abitanti per Regione',
    labels={
        'imprese_per_milione_di_abitanti': 'Imprese per Milione di Abitanti',
        'regione': 'Regione'
    },
    color='imprese_per_milione_di_abitanti',
    color_continuous_scale='Blues'
)

# Miglioriamo il layout
fig.update_layout(
    height=700,
    width=900,
    showlegend=False,
    title_x=0.5,
    xaxis_gridcolor='LightGray'
)

# Mostriamo il grafico
fig.show()

---
## Compito a casa

- [Kaggle Learn – Pandas](https://www.kaggle.com/learn/pandas), lezione **Renaming and Combining** (~20 min)
- Facoltativo: [W3Schools – Pandas](https://www.w3schools.com/python/pandas/default.asp), per ripassare